# Cargo Hi5 — Branch B CGR Core + Renderers

Recommended notebook for Cargo runtime. This notebook clones `MinhBe/GAN_SQLi`, installs the package, runs tests, detects GPU/runtime paths, runs a smoke pass, then runs Branch B full training if `RUN_FULL=True`.

Branch B: train Y1/Y2 CGR core generators, then render Y3/Y4 from accepted core candidates.


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/MinhBe/GAN_SQLi.git'
REPO_BRANCH = 'main'
WORK_ROOT = Path('/workspace') if Path('/workspace').exists() else Path.cwd()
REPO_DIR = WORK_ROOT / 'GAN_SQLi'
RUN_ROOT = WORK_ROOT / 'sqlgan_runs'

# Cargo control flags
RUN_TESTS = True
RUN_SMOKE = True
RUN_FULL = True
FORCE_RECLONE = True

# Full Branch B configuration
SURFACE = 'validation'
FULL_MLE_EPOCHS = 12
FULL_D_EPOCHS = 4
FULL_ADV_EPOCHS = 4
FULL_ADV_STEPS = 100
FULL_ROLLOUTS = 4
FULL_BATCH_SIZE = 256
FULL_GENERATE_N = 5000
FULL_PER_PARENT = 2
MAX_LEN = 192

# Smoke configuration. Keep small; this validates clone/install/data/code path.
SMOKE_ADV_EPOCHS = 1
SMOKE_OUT = RUN_ROOT / 'branch_b_smoke'
FULL_OUT = RUN_ROOT / 'branch_b_full'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('WORK_ROOT =', WORK_ROOT)
print('REPO_DIR  =', REPO_DIR)
print('RUN_ROOT  =', RUN_ROOT)


In [ ]:
import os, sys, subprocess, shutil, platform, json

def run(cmd, cwd=None, check=True):
    print('\n$ ' + ' '.join(map(str, cmd)))
    p = subprocess.run(list(map(str, cmd)), cwd=cwd, text=True)
    if check and p.returncode != 0:
        raise SystemExit(p.returncode)
    return p.returncode

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))
except Exception as e:
    print('Torch import before install failed:', repr(e))


In [ ]:
if FORCE_RECLONE and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR)


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=REPO_DIR)
run([sys.executable, '-m', 'pip', 'install', '-e', '.'], cwd=REPO_DIR)
if RUN_TESTS:
    run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR)


In [ ]:
# Dataset auto-detection. Prefer the cloned repo root because V4.1 dataset is stored in this repo.
candidates = [
    REPO_DIR,
    REPO_DIR / 'SQLGAN_PostgreSQL_Boolean_Attack_Corpus_V4_1_StaticTrainingCorpus',
    WORK_ROOT,
]
DATA_PATH = None
for c in candidates:
    if (c / 'A_generator_attack_corpus').exists() or (c / 'Dataset' / 'A_generator_attack_corpus').exists():
        DATA_PATH = c
        break
if DATA_PATH is None:
    zips = list(WORK_ROOT.rglob('*V4*StaticTrainingCorpus*.zip')) + list(WORK_ROOT.rglob('*SQLGAN*Boolean*.zip'))
    if zips:
        DATA_PATH = zips[0]
if DATA_PATH is None:
    raise FileNotFoundError('Cannot find SQLGAN dataset. Put dataset folder/zip in repo or Cargo workspace.')
print('DATA_PATH =', DATA_PATH)


In [ ]:
# Length audit before training. Increase MAX_LEN if truncation_rate_at_max_len > 0.
audit_json = RUN_ROOT / 'length_audit_branch_b.json'
run([sys.executable, '-m', 'sqlgan_dual.audit_length', '--data', str(DATA_PATH), '--out-json', str(audit_json), '--surface', SURFACE, '--max-len', str(MAX_LEN)], cwd=REPO_DIR)
print(audit_json.read_text()[:4000])


In [ ]:
if RUN_SMOKE:
    run([sys.executable, '-m', 'sqlgan_dual.experiment_b_core_renderers',
         '--data', str(DATA_PATH), '--out', str(SMOKE_OUT),
         '--surface', SURFACE, '--smoke', '--adv-epochs', str(SMOKE_ADV_EPOCHS),
         '--max-len', str(MAX_LEN)], cwd=REPO_DIR)
    print('Smoke output:', SMOKE_OUT)


In [ ]:
if RUN_FULL:
    run([sys.executable, '-m', 'sqlgan_dual.experiment_b_core_renderers',
         '--data', str(DATA_PATH), '--out', str(FULL_OUT),
         '--surface', SURFACE,
         '--mle-epochs', str(FULL_MLE_EPOCHS),
         '--d-epochs', str(FULL_D_EPOCHS),
         '--adv-epochs', str(FULL_ADV_EPOCHS),
         '--adv-steps', str(FULL_ADV_STEPS),
         '--rollouts', str(FULL_ROLLOUTS),
         '--batch-size', str(FULL_BATCH_SIZE),
         '--generate-n', str(FULL_GENERATE_N),
         '--per-parent', str(FULL_PER_PARENT),
         '--max-len', str(MAX_LEN)], cwd=REPO_DIR)
    print('Full output:', FULL_OUT)


In [ ]:
# Summarize outputs
for p in [SMOKE_OUT / 'branch_b_summary.json', FULL_OUT / 'branch_b_summary.json', RUN_ROOT / 'length_audit_branch_b.json']:
    if p.exists():
        print('\n====', p, '====')
        print(p.read_text()[:8000])
